# Required Submission Workflow: Torch Model

This notebook is intentionally simple. The model is a tiny PyTorch MLP. The important part is the required submission shape: reusable feature-building, reusable prediction, and MLflow `model_submission/` logging. There is no required load helper; the required pieces are the model architecture, the feature builder, the prediction function, and the saved artifact/metadata.

## 1. Imports And Run Configuration

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from torch import nn

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
model_name = 'torch_required_submission_example'
horizon = 5
target_col = f'forward_return_{horizon}d'
output_dir = repo_root / 'runs' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
print('Repo root:', repo_root)
print('Dataset:', dataset_name, spec.name)
print('Model name:', model_name)

## 2. Required Function: Build Model Features

This function is required. It must contain every feature used at inference time, including custom features. Adam can call this same function later on a private dataset.

In [ ]:
base_feature_names = [
    'momentum_20d',
    'vol_20d',
    'rsi_14',
    'price_to_sma_20d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
]

custom_feature_names = [
    'mom_vol_ratio',
]

feature_names = base_feature_names + custom_feature_names

def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    features = build_features(prices, feature_names=base_feature_names)
    features['mom_vol_ratio'] = features['momentum_20d'] / features['vol_20d'].replace(0.0, np.nan)
    return features.replace([np.inf, -np.inf], np.nan)

print('Feature names in model order:')
for feature in feature_names:
    print(' ', feature)

## 3. Required Class: Model Architecture

For Torch submissions, the notebook must define the exact class needed to load the saved `state_dict`.

In [ ]:
class TinyTorchReturnModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

## 4. Train The Simple Model

This section is deliberately basic. It trains on the public train split and uses train-only mean/std scaling.

In [ ]:
prices = load_prices(dataset_name, repo_root=repo_root)
features = build_model_features(prices)
targets = make_forward_return_target(prices, horizon=horizon)

model_frame = (
    features
    .merge(targets[['date', 'ticker', target_col]], on=['date', 'ticker'], how='left')
    .dropna(subset=feature_names + [target_col])
    .reset_index(drop=True)
)

train = slice_split(model_frame, dataset_name, 'train', repo_root=repo_root)
test = slice_split(model_frame, dataset_name, 'test', repo_root=repo_root)

train_means = train[feature_names].mean()
train_stds = train[feature_names].std(ddof=0).replace(0.0, 1.0)

def transform_features(frame: pd.DataFrame) -> np.ndarray:
    return ((frame[feature_names] - train_means) / train_stds).to_numpy(dtype=np.float32)

X_train = torch.tensor(transform_features(train), dtype=torch.float32)
y_train = torch.tensor(train[target_col].to_numpy(dtype=np.float32), dtype=torch.float32)

torch.manual_seed(42)
model = TinyTorchReturnModel(input_dim=len(feature_names), hidden_dim=8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(10):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()

print('Final train loss:', float(loss.item()))

## 5. Required Function: Save Submission Model Artifact

For Torch, save the `state_dict` plus metadata needed to recreate preprocessing and architecture.

In [ ]:
model_path = output_dir / 'torch_model.pt'
submission_metadata = {
    'feature_names': feature_names,
    'train_means': train_means.to_dict(),
    'train_stds': train_stds.to_dict(),
    'config': {
        'input_dim': len(feature_names),
        'hidden_dim': 8,
        'horizon': horizon,
        'target': target_col,
    },
}
submission_checkpoint = {
    'model_state_dict': model.state_dict(),
    **submission_metadata,
}
torch.save(submission_checkpoint, model_path)

print('Saved Torch model artifact:', model_path)

## 7. Required Function: Predict From Prices

This is the most important reproducibility function. It takes a price frame, rebuilds features, applies the submission metadata created above, and returns the standard prediction contract. In this notebook it uses the in-memory trained model. Later, Adam can manually recreate the same model class, load the saved `state_dict`, and pass the loaded model plus the checkpoint metadata into this same function.

In [ ]:
def predict_from_prices(model, metadata: dict, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    model.eval()
    features = build_model_features(prices)
    if dates is not None:
        dates = pd.to_datetime(dates)
        features = features.loc[features['date'].isin(dates)].copy()
    if tickers is not None:
        tickers = [ticker.upper() for ticker in tickers]
        features = features.loc[features['ticker'].isin(tickers)].copy()

    feature_order = metadata['feature_names']
    means = pd.Series(metadata['train_means'])
    stds = pd.Series(metadata['train_stds']).replace(0.0, 1.0)
    scoring_frame = features.dropna(subset=feature_order).reset_index(drop=True)
    X = ((scoring_frame[feature_order] - means[feature_order]) / stds[feature_order]).to_numpy(dtype=np.float32)

    with torch.no_grad():
        scores = model(torch.tensor(X, dtype=torch.float32)).numpy()

    predictions = scoring_frame[['date', 'ticker']].copy()
    predictions['horizon'] = int(metadata['config']['horizon'])
    predictions['expected_return'] = scores
    return predictions

raw_predictions = predict_from_prices(model, submission_metadata, prices)
test_dates = pd.to_datetime(test['date'].unique())
raw_predictions = raw_predictions.loc[raw_predictions['date'].isin(test_dates)].copy()
predictions = validate_prediction_frame(raw_predictions, dataset_name=dataset_name, horizon=horizon, repo_root=repo_root)
display(predictions.head())

## 8. Backtest The Submitted Predictions

In [ ]:
portfolio = weights_from_predictions_rank_long_only(predictions, dataset_name=dataset_name, strategy_name=model_name)
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=dataset_name, repo_root=repo_root)
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
artifact_paths = write_backtest_artifacts(result, output_dir)

print('Weights:', validated_weights.shape)
print('Metrics:')
for key, value in sorted(build_metrics(result).items()):
    print(f'  {key}: {value:.6f}')

## 9. Required MLflow Submission Logging

This logs normal research artifacts plus the reconstructable `model_submission/` bundle.

In [ ]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=f'{model_name}_submission',
    dataset_name=dataset_name,
    tags={
        'workflow': 'required_torch_submission',
        'model_family': 'torch',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        'model_name': model_name,
        'dataset_name': dataset_name,
        'horizon': horizon,
        'feature_count': len(feature_names),
        'feature_list': ','.join(feature_names),
        'target': target_col,
        'portfolio_builder': 'weights_from_predictions_rank_long_only',
    })

    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

    manifest = log_model_submission(
        {'torch_model': model_path},
        model_name=model_name,
        model_family='torch',
        feature_names=feature_names,
        target=target_col,
        horizon=horizon,
        preprocessing={
            'scaler': 'train_mean_std',
            'train_means': train_means.to_dict(),
            'train_stds': train_stds.to_dict(),
        },
        model_config={
            'architecture': 'TinyTorchReturnModel',
            'input_dim': len(feature_names),
            'hidden_dim': 8,
            'portfolio_builder': 'weights_from_predictions_rank_long_only',
            'required_functions': ['build_model_features', 'predict_from_prices'],
        },
        source_files=[repo_root / 'notebooks' / 'templates' / 'torch_required_submission_workflow.ipynb'],
        notes='Required Torch submission example with reusable feature and predict functions. No required load helper.',
    )

print(json.dumps(manifest, indent=2))